In [ ]:
import pandas as pd
import os
import glob
import datetime
import json
import re
import tqdm
import requests
from requests.exceptions import HTTPError

In [ ]:
#Library for social network analysis
import networkx as nx
import networkx.algorithms.community as nx_comm
from networkx.algorithms import bipartite
#import bamboolib
import pelote
from ipysigma import Sigma
import numpy as np
#import plotly.express as px


# Chargement des fichiers de données

On part du fichier fournit par le MESR qui est au format jsonl. On veut le passer en csv pour faciliter les manipulations

In [ ]:
file_path = "../Data/fichier_from_MESR/bso-publications-latest_199318270_enriched.jsonl"
list_id_pub = []
datas =[]
with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            data = json.loads(line)
            # Extract specific fields
            ids = data.get('all_ids')
            list_id_pub.append(ids[0])
            datas.append(data)
            
        except json.JSONDecodeError:
            continue  # Skip invalid lines


In [ ]:
datas[0]

In [ ]:
re.match('hal', 'hal-1')

In [ ]:
authors_halid_s = []
authors_halid_i = []
authors_fullname = []
authors_orcid = []
structures_id = []
structures_country = []
hal_class_code = []
bso_class = []
pub_year = []
pub_date = []
pub_type = []
pub_hal_id = []
is_oa = []
list_ids = []

for n, x in enumerate(list_id_pub):
    
    dict_pub = datas[n]
    if 'authors' in dict_pub:
        #print(n, x)
        list_author = []
        list_author_id_s = []
        list_author_id_i = []
        list_author_orcid = []
        list_structures_id = []
        list_country = []
        list_keyword = []
        list_ids.append(x)
        
        for a in dict_pub['authors']:
            #print(a)
            
            if  'full_name' in a:
                list_author.append(a['full_name'])
            else:
                list_author.append('no full_name')
            if 'id_hal_s' in a:
                list_author_id_s.append(a['id_hal_s'])
            else:
                list_author_id_s.append('no_id_hal_s')
            if 'id_hal_i'  in a:
                list_author_id_i.append(a['id_hal_i'])
            else:
                list_author_id_i.append('no_id_hal_i')
            if 'orcid' in a:
                list_author_orcid.append(a['orcid'])
            else:
                list_author_orcid.append('no orcid')
            if "affiliations" in a:
                for affil in a["affiliations"][0]:
                    print(affil)
    
        authors_halid_s.append("|".join(list_author_id_s))
        authors_halid_i.append("|".join(list_author_id_i))
        authors_fullname.append("|".join(list_author))
        authors_orcid.append("|".join(list_author_orcid))
        #structures_country.append("|".join(list_country))

        if 'hal' in x:
            pub_hal_id.append(x[3:])
        else:
            pub_hal_id.append(np.nan)
        if 'hal_classification' in dict_pub:
            hal_classification = [h['code'] for h in dict_pub['hal_classification']]
            hal_class_code.append('|'.join(hal_classification))
        else:
            hal_class_code.append('no hal_classification')
        bso_class.append(dict_pub['bso_classification'])
        if 'keywords' in dict_pub:
            for kw in dict_pub["keywords"]:
                list_keyword.append(kw["keyword"])
        else:
            list_keyword.append('no keywords')
        affiliations_i = [inst for inst in dict_pub['bso_local_affiliations']  if re.search(r'[a-z]', inst) is None ]
        structures_id.append('|'.join(affiliations_i))
        pub_type.append(dict_pub['genre'])
        #pub_date.append(dict_pub['published_date'])
        pub_year.append(dict_pub['year'])

        for y in dict_pub['oa_details']:
            if y == "2024Q4":
                dict_oa = dict_pub['oa_details'][y]
                #published_date.append(dict_pub['external_ids']['published_date'])
                is_oa.append(dict_oa["is_oa"])
    else:
        pass
        
    
        

    

In [ ]:
data_columns = {
    "id": list_ids,
    'hal_id': pub_hal_id,
    "pub_year" : pub_year,
    'authors' : authors_fullname,
    'authhalid_s': authors_halid_s,
    'authhalid_i':authors_halid_i,
    'orcid': authors_orcid,
    'struchal_id': structures_id,
    'struc_country': structures_country,
    'hal_classif' : hal_class_code,
    'bso_classif' : bso_class,
    'pub_type': pub_type,
    'is_oa': is_oa
}

df0 = pd.DataFrame(data= data_columns)


In [ ]:
df0.to_csv("../Data/outputs/2026-01-23_list_of_publi_from_bso.csv", sep = ",", index = False)

In [ ]:
import requests
from requests.exceptions import HTTPError

In [ ]:
import hal_extractor

In [ ]:
list_idhal = hal_extractor.get_list_idhal("../Data/outputs/2026-01-23_list_of_publi_from_bso.csv",  column = 'authhalid_i')
list_idhal2 = [x for x in list_idhal if re.search(r'[a-z]',x) is None]
len(list_idhal2)

In [ ]:
url_api_search = "https://api.archives-ouvertes.fr/ref/author/"


params = {
    "q":f"idHal_i:965723", #la clé de recherche. On peut envisager de faire une recherche sur d'autre champs
    "wt":"json", #Le format de sortie, ici json
    "fl" : "idHal_i,idHal_s,lastName_s,firstName_s,affPref_i,accountAssociated_bool,form_i,hasCV_bool", #La liste des champs obtenues
    "rows" : 1} # le nombre de ligne. Par défaut, l'api affiche les 30 première ligne et le maximum fixé est de 10000.

In [ ]:
hal_extractor.try_get_answer(url_api_search, params).json()

In [ ]:

path_folder = hal_extractor.create_folder(ref= "auteur", year=2026)
print(path_folder)
for n, idhal in tqdm.tqdm(enumerate(list_idhal2), total = len(list_idhal)):
    params["q"]= f"idHal_i:{idhal}"
    reponse = hal_extractor.try_get_answer(url_api_search, params).json()
    list_row = hal_extractor.process_json_to_csv(reponse,  params,  ref = "auteur", path_folder=path_folder)

In [ ]:
list_idhal = hal_extractor.get_list_idhal("../Data/raw/auteur/2025/hal_auteur.csv", column = 'affPref_i')

In [ ]:
url_api_search = "https://api.archives-ouvertes.fr/ref/structure/"


params = {
    "q":f"docid:11141", #la clé de recherche. On peut envisager de faire une recherche sur d'autre champs
    "wt":"json", #Le format de sortie, ici json
    "fl" : "docid,name_s,type_s,acronym_s,country_s,parentAcronym_s,parentDocid_i,parentName_s,parentType_s", #La liste des champs obtenues
    "rows" : 1} # le nombre de ligne. Par défaut, l'api affiche les 30 première ligne et le maximum fixé est de 10000.
for y in dict_pub['oa_details']:
        dict_oa = dict_pub['oa_details'][y]
        list_ids.append(x)
        #published_date.append(dict_pub['external_ids']['published_date'])
        pub_year.append(str(dict_pub['year']))
        is_oa.append(dict_oa["is_oa"])

hal_extractor.try_get_answer(url_api_search, params).json()

In [ ]:
path_folder = hal_extractor.create_folder(ref = "structure", year = 2026)
for n, idhal in tqdm.tqdm(enumerate(list_idhal), total = len(list_idhal)):
    params["q"]= f"docid:{idhal}"
    reponse = requests.get(url_api_search, params).json()
    #print(reponse)
    list_row = hal_extractor.process_json_to_csv(reponse, params, ref = "structure", path_folder = path_folder)

# Create liste of edges

In [ ]:
#df = pd.read_csv("/home/aymeric/paris8/barometre_scienceouverte_universitedelorraine/Data/outputs/20250507_hal_univ_paris8.csv", sep =",") # contient les publications
dfs = pd.read_csv("../Data/raw/structure/2026/hal_structure.csv", sep =",", dtype={"docid":str}) # Contient les structures
dfa = pd.read_csv("../Data/raw/auteur/2025/hal_auteur.csv", sep =",", dtype={"idHal_i":str}) # Contient les auteurs
df0.loc[df0.authhalid_s.str.contains("luneau")]
dfs.country_s.value_counts()

In [ ]:
dfa['docid'] = dfa.affPref_i.str.split("|") #docid correspond au nom de la colonne contenant les id des structures dans dfs
dfa1 = dfa.explode("docid")
dfa2 = dfa1[['idHal_i', 'idHal_s','docid']].dropna()

dfas0 = dfa2.merge(dfs, on = ["docid"], how = "left")
dfas = dfas0.loc[~dfas0.name_s.isna()].rename(columns={"idHal_i":"id"})
dfas["struct_id"] = dfas["docid"] +"_str"
dfas["author_id"] = dfas["id"] +"_au"


In [ ]:
df.loc[df.idHal_s.str.lower().str.contains("luneau")]

On "explose" le dataframe concernant les publi pour avoir une ligne par co-auteur. permets de faire ensuite la liste des edges

In [ ]:
df["author_id"] = df.authIdHal_i.str.split("|")
df1 = df.explode("author_id")



liste_halid_i = [x for x in df1[["author_id"]].author_id.drop_duplicates() if x is not np.nan]
df2 = df1[["halId_s", "author_id"]].dropna().sort_values("halId_s")
df2["author_id"] = df2["author_id"] +"_au"
df2.loc[df2.author_id=="965723_au"]


In [ ]:
df.loc[df.authFullName_s.str.contains("Luneau")]

In [ ]:
dfalab = dfas.loc[dfas.type_s=="laboratory"].rename(columns={"docid":"labhal_id",'acronym_s':'lab_acronym',"name_s":"lab_name"})#.dropna()
dfauniv = dfas.loc[dfas.type_s=="institution"].rename(columns={"docid":"instithal_id",'acronym_s':'instit_acronym', "name_s":"instit_name"})#.dropna()
dfas.loc[dfas.author_id=="965723_au"]
#df2b.loc[~(df2b.id.isna())|~(df2b.idHal_s.isna())]

In [ ]:
df2b = df2.merge(dfalab, on =["author_id"], how ="left").dropna()


In [ ]:
df2b

In [ ]:
dict_openaccess = dict(zip(df.halId_s, df.openAccess_bool))
df2["is_oa"] = df2.halId_s.map(dict_openaccess.get)
freq_oa = df2.groupby(["author_id","is_oa"]).agg(nb=("halId_s", "size" )).reset_index()
freq_total = df2.groupby(["author_id"]).agg(total=("halId_s", "size" )).reset_index()
freq_oa =freq_oa.merge(freq_total, on = ["author_id"], how = "left")
freq_oa["ratio_oa"] = (freq_oa.nb / freq_oa.total)
freq_oa["log_ratio_oa"] = np.log(1 + freq_oa.nb)
freq_oa["log_ratio_oa1"] = freq_oa["log_ratio_oa"].astype(int)
freq_oa["log_ratio_oa2"] = freq_oa["log_ratio_oa"].round(1)#astype(int)

dict_freq_oa = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.nb.loc[freq_oa.is_oa==True]))
dict_total = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.total.loc[freq_oa.is_oa==True]))
dict_ratio_oa = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.ratio_oa.loc[freq_oa.is_oa==True]))
dict_log_oa = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa.loc[freq_oa.is_oa==True]))
dict_log_oa2 = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa1.loc[freq_oa.is_oa==True]))
dict_log_oa3 = dict(zip(freq_oa.author_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa2.loc[freq_oa.is_oa==True]))
freq_oa

In [ ]:
df_halid = df2b[["halId_s"]].rename(columns={"halId_s":"id"}).drop_duplicates()
df_halid["Type"] = "publi"
df_auth= df2b[["author_id","idHal_s","lab_acronym","parentAcronym_s", "parentName_s"]].drop_duplicates("author_id").rename(columns={"author_id":"id"})
df_auth["Type"] = "auteur"
df_auth["nb_oa"] = df_auth.id.map(dict_freq_oa.get)
df_auth["ratio_oa"] = df_auth.id.map(dict_ratio_oa.get)
df_auth["log_ratio_oa"] = df_auth.id.map(dict_log_oa.get)
df_auth["log_ratio_oa1"] = df_auth.id.map(dict_log_oa2.get)
df_auth["log_ratio_oa2"] = df_auth.id.map(dict_log_oa3.get)
df_auth["total"] = df_auth.id.map(dict_total.get)
df_auth = df_auth.fillna(0)




## Rattachement institutionnel des labos à P8

for

In [ ]:
"parentAcronym_s", "parentName_s"
df_auth.loc[df_auth.parentAcronym_s.str.lower().str.contains("up8") == True, "is_P8"] = True
df_auth.loc[df_auth.parentAcronym_s.str.lower().str.contains("up8") == False, "is_P8"] = False

nodes = pd.concat([df_halid, df_auth])
nodes

# Graphe Auteur-Publi

In [ ]:
G = nx.Graph()
G.add_nodes_from(df2b.halId_s, part="publi")
G.add_nodes_from(df2b.author_id, part="auteur")
G.add_edges_from(df2b[["halId_s", "author_id"]].values)



In [ ]:
nodes["Label"] = nodes["id"]
node_attr = nodes.set_index('id').to_dict('index')
nx.set_node_attributes(G, node_attr)



In [ ]:
Sigma(G, node_color="is_P8")

In [ ]:
monoG = pelote.monopartite_projection(G, 'auteur')
nx.number_of_nodes(monoG)
nx.number_connected_components(monoG)

df_auth["Label"] = df_auth["id"]
node_attr = df_auth.set_index('id').to_dict('index')
nx.set_node_attributes(monoG, node_attr)


In [ ]:
degres = nx.degree_centrality(monoG)

not_normalized_degres = {}

for x in degres :
    not_normalized_degres[x] = degres[x] * (nx.number_of_nodes(monoG)-1)


nx.set_node_attributes(monoG, degres, 'normalized_degres')
nx.set_node_attributes(monoG, not_normalized_degres, 'degres')
#h = pelote.filter_nodes(monoG, lambda n, a: a["degres"] > 2)
#pelote.crop_to_largest_connected_component(monoG)
print(nx.number_connected_components(monoG))
louv_communities = nx.community.louvain_communities(monoG)

community={}

print(len(louv_communities))
for n, x in enumerate(louv_communities):
    #print(n)
    for name in x:
        community[name] = n

nx.set_node_attributes(monoG, community, 'community')   


In [ ]:
Sigma(monoG, node_label='idHal_s', node_color= "is_P8", node_shape="type_node", node_size="nb_oa")


# Graphe Publi-labo

In [ ]:
df2bb = df2b[["halId_s","labhal_id", "lab_acronym","parentAcronym_s", "parentName_s"]].drop_duplicates()
G = nx.Graph()
G.add_nodes_from(df2bb.halId_s, part="publi")
G.add_nodes_from(df2bb.labhal_id, part="labos")
G.add_edges_from(df2b[["halId_s", "labhal_id"]].values)

In [ ]:
dict_openaccess = dict(zip(df.halId_s, df.openAccess_bool))
df2bb["is_oa"] = df2bb.halId_s.map(dict_openaccess.get)
freq_oa = df2bb.groupby(["labhal_id","is_oa"]).agg(nb=("labhal_id", "size" )).reset_index()
freq_total = df2bb.groupby(["labhal_id"]).agg(total=("labhal_id", "size" )).reset_index()
freq_oa =freq_oa.merge(freq_total, on = ["labhal_id"], how = "left")
freq_oa["ratio_oa"] = (freq_oa.nb / freq_oa.total)
freq_oa["log_ratio_oa"] = np.log(1 + freq_oa.nb)
freq_oa["log_ratio_oa1"] = freq_oa["log_ratio_oa"].astype(int)
freq_oa["log_ratio_oa2"] = freq_oa["log_ratio_oa"].round(1)#astype(int)

dict_freq_oa = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.nb.loc[freq_oa.is_oa==True]))
dict_total = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.total.loc[freq_oa.is_oa==True]))
dict_ratio_oa = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.ratio_oa.loc[freq_oa.is_oa==True]))
dict_log_oa = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa.loc[freq_oa.is_oa==True]))
dict_log_oa2 = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa1.loc[freq_oa.is_oa==True]))
dict_log_oa3 = dict(zip(freq_oa.labhal_id.loc[freq_oa.is_oa==True], freq_oa.log_ratio_oa2.loc[freq_oa.is_oa==True]))
freq_oa

In [ ]:
df_halid = df2bb[["halId_s"]].rename(columns={"halId_s":"id"}).drop_duplicates()
df_halid["Type"] = "publi"
df_auth= df2bb[["labhal_id", "lab_acronym","parentAcronym_s", "parentName_s"]].drop_duplicates().rename(columns={"labhal_id":"id"})
df_auth["Type"] = "labos"

df_auth.loc[df_auth.parentAcronym_s.str.lower().str.contains("up8") == True, "is_P8"] = True
df_auth.loc[df_auth.parentAcronym_s.str.lower().str.contains("up8") == False, "is_P8"] = False


df_auth["nb_oa"] = df_auth.id.map(dict_freq_oa.get)
df_auth["ratio_oa"] = df_auth.id.map(dict_ratio_oa.get)
df_auth["log_ratio_oa"] = df_auth.id.map(dict_log_oa.get)
df_auth["log_ratio_oa1"] = df_auth.id.map(dict_log_oa2.get)
df_auth["log_ratio_oa2"] = df_auth.id.map(dict_log_oa3.get)
df_auth["total"] = df_auth.id.map(dict_total.get)

nodes = pd.concat([df_halid, df_auth])
df_auth

In [ ]:
nodes["Label"] = nodes["id"]
node_attr = nodes.set_index('id').to_dict('index')
nx.set_node_attributes(G, node_attr)



In [ ]:
monoG = pelote.monopartite_projection(G, 'labos')
nx.number_of_nodes(monoG)
nx.number_connected_components(monoG)

df_auth["Label"] = df_auth["id"]
node_attr = df_auth.set_index('id').to_dict('index')
nx.set_node_attributes(monoG, node_attr)


In [ ]:
degres = nx.degree_centrality(monoG)

not_normalized_degres = {}

for x in degres :
    not_normalized_degres[x] = degres[x] * (nx.number_of_nodes(monoG)-1)


nx.set_node_attributes(monoG, degres, 'normalized_degres')
nx.set_node_attributes(monoG, not_normalized_degres, 'degres')
#h = pelote.filter_nodes(monoG, lambda n, a: a["degres"] > 2)
pelote.crop_to_largest_connected_component(monoG)
print(nx.number_connected_components(monoG))
louv_communities = nx.community.louvain_communities(monoG)

community={}

print(len(louv_communities))
for n, x in enumerate(louv_communities):
    #print(n)
    for name in x:
        community[name] = n

nx.set_node_attributes(monoG, community, 'community')   


In [ ]:
Sigma(monoG, node_label='lab_acronym', node_color= "is_P8", node_shape="Type", node_size="nb_oa")


## Graphe auteur et institution

In [ ]:

dfasb = dfas.loc[(dfas.type_s=="laboratory")]
dfasb.loc[(dfasb.acronym_s.isna()), "acronym"] = dfasb.docid
dfasb.loc[~(dfasb.acronym_s.isna()), "acronym"] = dfasb.acronym_s
dfasb.loc[(dfasb.idHal_s.isna()), "idHal"] = dfasb.id
dfasb.loc[~(dfasb.idHal_s.isna()), "idHal"] = dfasb.idHal_s
dfas_str = dfasb[["acronym","type_s"]].rename(columns={"acronym":"nom"}).drop_duplicates()

dfas_str["Type"] = "structure"
dfas_au= dfasb[["idHal"]].rename(columns={"idHal":"nom"}).drop_duplicates()

dfas_au["Type"] = "chercheur"
dfas_au["type_s"] = "chercheur"
nodes = pd.concat([dfas_au, dfas_str]).dropna()
nodes
#nodes = nodes0.merge(dfalab[['id', 'idHal_s', 'labhal_id', 'lab_name','lab_acronym']], on =["id"], how ="left")#.merge(dfauniv[['id', 'instithal_id', 'instit_name',
       #'instit_acronym']], on =["id"], how ="left")
#nodes = nodes0.merge(dfauniv[['id', 'idHal_s','instithal_id', 'instit_name', 'instit_acronym']], on =["id"], how ="left")
l = [x for x in dfasb.idHal if isinstance(x, float)==True]
nodes

In [ ]:
G = nx.Graph()
G.add_nodes_from(dfasb.acronym, part="structure")
G.add_nodes_from(dfasb.idHal, part="chercheur")
G.add_edges_from(dfasb[["idHal", "acronym"]].values)

In [ ]:
nodes["Label"] = nodes["nom"]
node_attr = nodes.set_index('nom').to_dict('index')
nx.set_node_attributes(G, node_attr)



In [ ]:
monoG = pelote.monopartite_projection(G, 'chercheur')
nx.number_of_nodes(monoG)
nx.number_connected_components(monoG)

In [ ]:
degres = nx.degree_centrality(monoG)

not_normalized_degres = {}

for x in degres :
    not_normalized_degres[x] = degres[x] * (nx.number_of_nodes(monoG)-1)


nx.set_node_attributes(monoG, degres, 'normalized_degres')
nx.set_node_attributes(monoG, not_normalized_degres, 'degres')
#h = pelote.filter_nodes(monoG, lambda n, a: a["degres"] > 2)
#pelote.crop_to_largest_connected_component(monoG)
print(nx.number_connected_components(monoG))
louv_communities = nx.community.louvain_communities(monoG)

community={}

print(len(louv_communities))
for n, x in enumerate(louv_communities):
    #print(n)
    for name in x:
        community[name] = n

nx.set_node_attributes(monoG, community, 'community')   


In [ ]:
Sigma(monoG, node_label="Label", node_color= "community", node_shape="type_s", node_size="degres")
